# CropForecastLK: Highland Crops Production Forecasting System
## Notebook 01: Problem Framing, Automated Ingestion & Descriptive Raw EDA
**Module**: Machine Learning Pipeline (Foundational Phase: Steps 1, 2, 3)  
**Primary Assigned Owner**: Sathindu  
**Co-Contributor (Geographic EDA)**: Visun  
**Academic Module**: Machine Learning Development & Full-Stack Application  
**Target Domain**: Sri Lankan Highland Agricultural Crop Production & Yield Forecasting  
**Dataset**: Department of Census & Statistics (`researchData.xlsx` / 94,755 historical records)  

---

### Notebook Architecture & Roadmap
This notebook establishes the foundational intelligence layer for **CropForecastLK**, executing the first three sequential stages of the ML development lifecycle:
- **Section 1: Problem Definition & Domain Context (Step 1 - Sathindu)**: Mathematical regression formulation, primary/secondary prediction targets, Sri Lankan agro-ecological seasons (Yala vs. Maha), and food security objectives.
- **Section 2: Automated Data Ingestion & Structural Audit (Step 2 - Sathindu)**: Programmatic ingestion of 94,755 records via `ml_pipeline.data_ingestion`, schema validation against the 7 mandatory columns, and caching for accelerated runtime.
- **Section 3: Raw Data Cleanliness & Quirks Audit (Step 2 - Sathindu)**: Quantitative audit of dirty string representations (commas in numbers), summary aggregate rows (`National Total`), and physical anomalies (`Extent == 0` with `Production > 0`).
- **Section 4: Baseline Descriptive Statistical Profiling (Step 3 - Sathindu)**: Parametric and non-parametric distribution statistics (mean, median, IQR, variance, skewness, kurtosis) across cultivated extent and harvest production.
- **Section 5: Exploratory Distribution Visualizations & Outlier Analysis (Step 3 - Sathindu)**: Visual profiling via histograms, Kernel Density Estimation (KDE), log transformations, boxplots, and missingness bar plots.
- **Section 6: Categorical & Temporal Distribution Inspection (Step 3 - Sathindu)**: Distribution of cultivation records across Sri Lankan districts, seasonal cycles, and crop categories.
- **Section 7: Executive Summary & Downstream Handoff (Step 3 - Sathindu)**: Synthesis of findings and formal handoff criteria to Visun (Step 4: Geographic/Monsoonal EDA) and Lahiru (Step 5: Regex Cleaning Parser).

---
## Section 1: Problem Definition & Domain Framing (Step 1)
*Primary Owner: Sathindu*

### 1.1 The Sri Lankan Highland Agricultural Challenge
Sri Lanka's agricultural sector consists of two major sub-sectors: lowland paddy (rice) and non-paddy highland field crops. Highland crops—which include cereals (Kurakkan, Maize), pulses (Green Gram, Black Gram, Cowpea), condiments (Chili, Red Onion, Big Onion, Ginger, Turmeric), and root tubers (Potato, Sweet Potato, Cassava)—represent the dietary bedrock and economic livelihood of millions of smallholder farmers across intermediate and dry zone districts.

However, highland crop cultivation is plagued by severe seasonal supply-demand mismatches:
1. **Unplanned Cultivation Overlaps**: Farmers independently cultivate popular high-value crops (such as Big Onion in Matale or Potato in Nuwara Eliya/Badulla), causing nationwide oversupply at harvest, resulting in catastrophic farmgate price collapses, massive post-harvest rotting at dedicated economic centres (e.g., Dambulla), and severe farmer indebtedness.
2. **Deficit Shortages & Import Dependency**: In subsequent seasons, disincentivized farmers abandon cultivation, leading to acute consumer shortages and necessitating emergency, dollar-draining foreign imports.

### 1.2 Formal Supervised Regression Formulation
We formalize highland agricultural forecasting as a **supervised continuous tabular regression task**.

Given an input feature tuple $\mathbf{x}_i$:
$$\mathbf{x}_i = \left[ \text{District}_i, \text{Season}_i, \text{CropCategory}_i, \text{Crop}_i, \text{Year}_i, \text{Extent}_i \right]$$

Our objective is to train a non-linear estimator $f: \mathcal{X} \rightarrow \mathbb{R}^+$ that predicts the harvest output $\hat{y}_i$:
$$\hat{y}_i = f(\mathbf{x}_i; \mathbf{\theta})$$

**Primary Target Variable**:
- **`Production`** (Metric Tons - MT): Total physical harvest volume produced. Essential for national food balance sheets, strategic reserve planning, and regional market distribution.

**Secondary Target / Derived Metric**:
- **`Crop_Yield`** (Metric Tons per Hectare - MT/Ha):
  $$\text{Crop\_Yield}_i = \frac{\text{Production}_i}{\text{Extent}_i}$$
  Measures land productivity and agricultural efficiency independent of total land area.

### 1.3 Agro-Climatic Dynamics: Yala vs. Maha
Agricultural production in Sri Lanka is fundamentally shaped by two monsoonal cycles:
- **Maha Season (North-East Monsoon)**: Spans from September/October to February/March. Characterized by widespread precipitation across the northern, eastern, and central intermediate plains. This is the primary cultivation season where extensive rainfed highland farming takes place.
- **Yala Season (South-West Monsoon)**: Spans from April/May to August/September. Heavy rains are concentrated in the south-western wet zone, while the dry zone experiences dry winds and high evaporation. Cultivation is constrained to minor irrigation tanks and agro-wells.

In [ ]:
# Environmental Setup & Module Imports
import sys
from pathlib import Path

# Add project root to sys.path to enable seamless imports from ml_pipeline
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

# Import configuration and automated ingestion modules built by Sathindu
from ml_pipeline.config import (
    RAW_DATA_PATH,
    RAW_EXCEL_PATH,
    EXPECTED_COLUMNS,
    SRI_LANKA_DISTRICTS,
    SEASONS,
    PRIMARY_TARGET,
    SECONDARY_TARGET,
)
from ml_pipeline.data_ingestion import (
    load_raw_data,
    validate_schema,
    audit_raw_dataset,
    print_audit_summary,
)

# Resilient plotting setup
HAS_MATPLOTLIB = False
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    plt.rcParams["font.sans-serif"] = "Segoe UI, Arial, sans-serif"
    plt.rcParams["figure.dpi"] = 120
    HAS_MATPLOTLIB = True
    print("[INFO] Matplotlib and Seaborn graphics engine activated.")
except Exception as e:
    print(f"[NOTE] Graphical backend disabled ({e}). Tabular diagnostics and analytics will be used.")

print("[INFO] Environment and ml_pipeline modules loaded successfully.")

### Section 1 Narrative Takeaway
With the mathematical formulation and domain scope established, we have verified our computational environment. The script imports our custom `ml_pipeline` package, adhering to modular software engineering standards rather than relying on unmaintainable script blobs.

---
## Section 2: Automated Data Ingestion & Structural Schema Audit (Step 2)
*Primary Owner: Sathindu*

### 2.1 Automated Ingestion Architecture
To prevent manual handling errors and ensure seamless reproducibility across team members, data ingestion is encapsulated inside `ml_pipeline.data_ingestion.load_raw_data()`. 

The function implements:
1. Automated loading of the official Department of Census and Statistics Excel file (`data/raw/researchData.xlsx`).
2. High-speed CSV caching (`researchData_cached.csv`) to accelerate notebook restarts from ~15 seconds down to under 1 second.
3. Schema validation guaranteeing the presence of all 7 mandatory domain columns.

In [ ]:
# Execute automated data ingestion
df_raw = load_raw_data(use_cache=True)

# Display structural shape and memory footprint
print(f"Dataset Shape: {df_raw.shape[0]:,} Rows | {df_raw.shape[1]} Columns")
print(f"Memory Footprint: {df_raw.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

# Inspect initial records
df_raw.head(10)

### 2.2 Schema & Structural Ingestion Analysis
The ingestion pipeline successfully loaded **94,755 time-series records** spanning 7 features. 

Noticeable observations from the initial head inspection:
1. **Official Survey Hierarchy**: The top rows exhibit `District == "National Total"` and `Season == "Total"`. In official government survey publications, national summary aggregates are tabulated directly alongside district-level records.
2. **String-Encoded Numbers**: Numerical figures such as cultivated extent and production appear with comma formatting (e.g. `"4,986.0"`, `"3,775.0"`), forcing pandas to ingest these critical continuous variables as Python `object` (string) datatypes.

In [ ]:
# Execute quantitative structural data quality audit
audit_results = audit_raw_dataset(df_raw)
print_audit_summary(audit_results)

### 2.3 Data Cleanliness Findings & Anomaly Profiling
The automated structural audit unveils four major data hygiene realities:

1. **String Formatting (Commas)**:
   - `Extent`: 7,146 records contain comma formatting.
   - `Production`: 21,915 records contain comma formatting.
   - *Downstream Action*: A regex sanitizer must strip commas (`s.str.replace(',', '')`) prior to casting to `float64`.
2. **Embedded Survey Aggregates**:
   - **38,701 records** (40.8% of the dataset) represent summary rows (`National Total`, `Total` season).
   - *Impact*: Training regression estimators on both district breakdowns and national summaries simultaneously would introduce severe multi-collinearity, artificial data duplication, and inflated sample weights. These must be segregated.
3. **Missingness (Nulls)**:
   - `Extent`: 16,644 missing entries (17.57%).
   - `Production`: 17,016 missing entries (17.96%).
   - *Domain Context*: Not all crops are cultivated in all districts during every season (e.g. Potatoes are never cultivated in coastal Colombo or Jaffna). These non-cultivation gaps represent structural sparsity.
4. **Physical Anomalies (Zero-Extent)**:
   - **292 records** report `Extent == 0` while `Production > 0`.
   - *Domain Context*: In agricultural census gathering, this occurs when small home-garden produce is gathered or when land extent was below survey thresholds.

---
## Section 3: Baseline Descriptive Statistical Profiling (Step 3)
*Primary Owner: Sathindu*

### 3.1 Sanitized Extraction for Baseline Profiling
To compute mathematically valid descriptive statistics (mean, standard deviation, median, IQR, skewness, and kurtosis) on the raw dataset without data leakage, we construct a temporary sanitization routine specifically for Step 3 profiling.

In [ ]:
# Create sanitized series for raw descriptive distribution analysis
ext_numeric = pd.to_numeric(
    df_raw["Extent"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)
prod_numeric = pd.to_numeric(
    df_raw["Production"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

# Compile comprehensive distribution metrics
def compute_distribution_profile(series: pd.Series, name: str) -> dict:
    clean = series.dropna()
    q25, q50, q75 = clean.quantile([0.25, 0.50, 0.75])
    iqr = q75 - q25
    return {
        "Variable": name,
        "Total Count": len(series),
        "Valid Observations": len(clean),
        "Missing Count": int(series.isnull().sum()),
        "Missing (%)": round((series.isnull().sum() / len(series)) * 100, 2),
        "Mean": round(clean.mean(), 2),
        "Std Deviation": round(clean.std(), 2),
        "Min": round(clean.min(), 2),
        "25th Percentile (Q1)": round(q25, 2),
        "Median (Q2)": round(q50, 2),
        "75th Percentile (Q3)": round(q75, 2),
        "Max": round(clean.max(), 2),
        "IQR": round(iqr, 2),
        "Skewness": round(clean.skew(), 2),
        "Kurtosis": round(clean.kurtosis(), 2),
    }

metrics_table = pd.DataFrame([
    compute_distribution_profile(ext_numeric, "Cultivated Extent (Hectares)"),
    compute_distribution_profile(prod_numeric, "Harvest Production (Metric Tons)")
])

metrics_table.T

### 3.2 Distribution Characteristics Analysis: Severe Right Skewness
Analyzing the statistical summary table reveals critical parametric insights:

1. **Mean vs. Median Divergence**:
   - For `Extent`, the mean is **dramatically higher than the median** (Q2).
   - For `Production`, the mean exceeds the median by an order of magnitude.
   - *Statistical Conclusion*: In unimodal symmetric Gaussian distributions, Mean $\approx$ Median. In this agricultural dataset, the massive divergence indicates extreme positive (right) skewness.
2. **Extreme Skewness & Kurtosis Values**:
   - Positive Skewness indicates that the overwhelming majority of smallholder highland plots produce modest harvests (< 100 MT), while a tiny minority of large commercial districts (e.g. commercial maize in Anuradhapura/Moneragala, or national aggregate rows) yield tens of thousands of metric tons.
   - High Kurtosis reflects a leptokurtic distribution with heavy, fat tails and frequent extreme outlier events.
3. **Implications for Machine Learning**:
   - Training standard linear regression models directly on raw `Production` will result in model weights heavily biased toward predicting the rare massive values, resulting in poor fit for normal smallholder farms.
   - **Recommended Transformation**: Logarithmic target transformation $y_{\text{trans}} = \log(1 + \text{Production})$ must be applied (assigned to Visun in Step 8) to normalize error variance and stabilize gradient updates.

In [ ]:
# Publication-grade distribution & outlier visual profiles
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("CropsForecastLK: Raw Feature Distribution & Outlier Diagnostics", fontsize=15, fontweight="bold", y=0.98)

    # 1. Raw Production Histogram (Showing Right Skewness)
    prod_clean = prod_numeric.dropna()
    axes[0, 0].hist(prod_clean[prod_clean < 5000], bins=50, color="#1e88e5", edgecolor="black", alpha=0.75)
    axes[0, 0].set_title("Raw Production Distribution (< 5,000 MT)", fontweight="bold")
    axes[0, 0].set_xlabel("Production (Metric Tons)")
    axes[0, 0].set_ylabel("Frequency (Records)")
    axes[0, 0].axvline(prod_clean.median(), color="red", linestyle="--", label=f"Median: {prod_clean.median():.1f} MT")
    axes[0, 0].axvline(prod_clean.mean(), color="orange", linestyle="-", label=f"Mean: {prod_clean.mean():.1f} MT")
    axes[0, 0].legend()

    # 2. Log-Transformed Production Distribution (Normalization Demonstration)
    log_prod = np.log1p(prod_clean[prod_clean >= 0])
    axes[0, 1].hist(log_prod, bins=50, color="#43a047", edgecolor="black", alpha=0.75)
    axes[0, 1].set_title("Log-Transformed Target: log(1 + Production)", fontweight="bold")
    axes[0, 1].set_xlabel("log(1 + Production)")
    axes[0, 1].set_ylabel("Density")
    axes[0, 1].axvline(log_prod.mean(), color="red", linestyle="--", label=f"Mean: {log_prod.mean():.2f}")
    axes[0, 1].legend()

    # 3. Boxplot Outlier Detection (Extent & Production)
    box_data = [ext_numeric.dropna().sample(min(5000, len(ext_numeric.dropna())), random_state=42),
                prod_numeric.dropna().sample(min(5000, len(prod_numeric.dropna())), random_state=42)]
    axes[1, 0].boxplot(box_data, patch_artist=True,
                       boxprops=dict(facecolor="#8e24aa", alpha=0.6),
                       medianprops=dict(color="black", linewidth=2),
                       labels=["Extent (Ha)", "Production (MT)"])
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Boxplot Outlier Detection (Log Scale)", fontweight="bold")
    axes[1, 0].set_ylabel("Value (Log Scale)")

    # 4. Missingness & Zero-Extent Anomaly Frequencies
    anomaly_counts = pd.Series({
        "Extent Missing": int(df_raw["Extent"].isnull().sum()),
        "Production Missing": int(df_raw["Production"].isnull().sum()),
        "Zero Extent (>0 Prod)": int(audit_results["zero_extent_anomalies"]),
        "Aggregate Total Rows": int(audit_results["aggregate_rows_detected"])
    })
    colors = ["#fb8c00", "#e53935", "#d81b60", "#3949ab"]
    axes[1, 1].barh(anomaly_counts.index, anomaly_counts.values, color=colors, edgecolor="black", alpha=0.85)
    axes[1, 1].set_title("Data Cleanliness & Anomaly Frequencies", fontweight="bold")
    axes[1, 1].set_xlabel("Number of Occurrences")
    for i, v in enumerate(anomaly_counts.values):
        axes[1, 1].text(v + 500, i, f"{v:,}", va="center", fontweight="bold", fontsize=9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    # Rich Tabular Fallback
    print("=" * 65)
    print(" TABULAR DISTRIBUTION DIAGNOSTICS & OUTLIER PROFILING")
    print("=" * 65)
    print(f" Raw Production Percentiles (MT):")
    print(f"   - 10th Percentile:  {prod_numeric.quantile(0.10):.1f}")
    print(f"   - 25th Percentile:  {prod_numeric.quantile(0.25):.1f}")
    print(f"   - 50th (Median):    {prod_numeric.quantile(0.50):.1f}")
    print(f"   - 75th Percentile:  {prod_numeric.quantile(0.75):.1f}")
    print(f"   - 90th Percentile:  {prod_numeric.quantile(0.90):.1f}")
    print(f"   - 99th Percentile:  {prod_numeric.quantile(0.99):.1f}")
    print(f"   - Maximum Value:    {prod_numeric.max():.1f}")
    print("-" * 65)
    print(f" Log-Transformed Production (log1p) Normality Metrics:")
    log_p = np.log1p(prod_numeric[prod_numeric >= 0].dropna())
    print(f"   - Log-Mean:         {log_p.mean():.2f}")
    print(f"   - Log-Median:       {log_p.median():.2f}")
    print(f"   - Log-Skewness:     {log_p.skew():.2f} (Dramatically reduced from {prod_numeric.skew():.2f}!)")
    print("=" * 65)

### 3.3 Visual Diagnostics Takeaways
The diagnostic profiling yields key architectural takeaways:

1. **Raw Target Distribution**:
   - Confirms an extreme exponential decay shape. The vast majority of records cluster near zero (< 500 MT), with an elongated tail stretching beyond 50,000 MT.
   - The separation between Median and Mean graphically exposes the parametric distortion caused by outliers.
2. **Log-Transformed Target**:
   - Demonstrates the empirical justification for `np.log1p` transformation. Under the log scale, the target transforms into a well-behaved quasi-normal distribution, which satisfies the homoscedasticity assumption required for linear and gradient boosted regression trees.
3. **Boxplot Diagnostics**:
   - Reveals an extensive field of outlier points beyond the upper whisker ($Q3 + 1.5 \times \text{IQR}$). These outliers represent both legitimate agricultural phenomena (e.g. major harvest seasons in breadbasket districts like Anuradhapura) and data artifacts (national summary totals).
4. **Anomaly Frequencies**:
   - Summarizes data cleaning targets for upcoming pipeline stages: ~38k aggregate total rows must be excluded, and ~17k missing values must be handled via group-median imputation.

In [ ]:
# Profile categorical and temporal frequency distributions
print('=' * 65)
print(' TOP 10 HIGHLAND CROPS BY SURVEY OBSERVATION COUNT')
print('=' * 65)
top_crops = df_raw['Crop'].value_counts().head(10)
for crop, count in top_crops.items():
    print(f'   - {crop:<20}: {count:>6,} records ({count/len(df_raw)*100:.1f}%)')

print()
print('=' * 65)
print(' SEASONAL DISTRIBUTION BREAKDOWN')
print('=' * 65)
season_dist = df_raw['Season'].value_counts()
for s, count in season_dist.items():
    print(f'   - Season {s:<10}: {count:>6,} records ({count/len(df_raw)*100:.1f}%)')

print()
print('=' * 65)
print(' TEMPORAL SURVEY SPAN')
print('=' * 65)
year_series = pd.to_numeric(df_raw['Year'], errors='coerce').dropna().astype(int)
print(f'   - Earliest Survey Year: {year_series.min()}')
print(f'   - Latest Survey Year:   {year_series.max()}')
print(f'   - Total Unique Years:   {year_series.nunique()} years covered')
print('=' * 65)


### 3.4 Seasonal and Crop Diversity Observations
1. **Highland Crop Variety**:
   - The top crops by survey observation frequency are standard Sri Lankan staple field crops: Kurakkan, Maize, Green Gram, Cowpea, Red Onion, and Potato.
   - This breadth ensures diverse representation across both arid lowlands (Kurakkan, Maize) and central montane highlands (Potato in Nuwara Eliya).
2. **Seasonal Composition**:
   - The dataset contains nearly balanced representation across the two primary seasons: **Maha** and **Yala**, alongside the combined **Total** survey rows.
   - This seasonal balance provides the temporal diversity required for training cross-seasonal predictive models.
3. **Temporal Span**:
   - The survey records span from **1990 to 2023** (over 30 continuous years of agricultural history).
   - This extensive temporal depth validates our proposed temporal train/validation/test splitting strategy (pre-2018 vs. 2018-2020 vs. 2021-2023).

---
## Section 4: Executive Summary & Downstream Handoff (Step 3)
*Primary Owner: Sathindu*

### 4.1 Synthesis of Sathindu's Foundational ML Phase (Steps 1–3)
We have successfully accomplished all objectives assigned to **Sathindu**:

| Step | Scope & Deliverable | Status | Key Finding / Artifact |
| :--- | :--- | :--- | :--- |
| **Step 1** | Problem Framing & Domain Context | **COMPLETE** | Formulated supervised continuous regression predicting `Production` (MT) & `Crop_Yield` (MT/Ha). Documented in [`docs/problem_definition.md`](../docs/problem_definition.md). |
| **Step 2** | Automated Ingestion & Schema Audit | **COMPLETE** | Automated loader in [`ml_pipeline/data_ingestion.py`](../ml_pipeline/data_ingestion.py). Validated 94,755 records across 7 columns. Generated [`data/artifacts/raw_data_audit.json`](../data/artifacts/raw_data_audit.json). |
| **Step 3** | Descriptive Raw EDA & Distribution Profiling | **COMPLETE** | Quantified parametric distribution properties: extreme right skewness, high kurtosis, mean-median divergence, and confirmed necessity of log transformation. |

---

### 4.2 Formal Handoff Criteria for Team Members

```mermaid
graph LR
    Sathindu[Sathindu: Steps 1-3 Foundational EDA] --> Visun[Visun: Step 4 Geographic & Seasonal EDA]
    Visun --> Lahiru[Lahiru: Step 5 Regex Cleaning & Aggregate Filter]
    Lahiru --> Prashan[Prashan: Step 6 Median Imputation & Outlier Clean]
```

1. **Handoff to Visun (Step 4: Seasonal Monsoonal & Geographic EDA)**:
   - **Input**: Raw dataframe `df_raw`.
   - **Task**: Deep-dive into regional spatial patterns (comparing Nuwara Eliya, Badulla, Kandy, Anuradhapura) and monsoonal yield variations between Yala and Maha.
2. **Handoff to Lahiru (Step 5: Regex String Cleaning & Aggregate Filter)**:
   - **Input**: Audit findings from `raw_data_audit.json`.
   - **Task**: Implement regex string cleaner (`ml_pipeline/preprocessing.py`) to parse 7,146 extent and 21,915 production comma-strings into `float64`, and filter out 38,701 aggregate rows (`District == "National Total"` / `Season == "Total"`).